# 🎯 [C202_M09] AAR 기반 중요 전문 전송 지연 사례 분석 및 큐 우선순위 가중치 재조정
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M09 |
| **미션 주제** | AAR 기반 중요 전문 전송 지연 사례 분석 및 큐 우선순위 가중치 재조정 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 중요 전문 전송 지연 사례 분석 및 큐 우선순위 가중치 재조정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR을 통해 중요 전문 전송 지연 사례를 분석하고 큐 스케줄링 가중치를 재조정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR로 중요 전문의 전송 지연 사례를 원인별로 분석하고 큐 스케줄링 가중치와 재전송 파라미터를 재조정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **초기 운용 설정(전문 단위 전송, 고정 재전송 타이머 6초, 조각 512bytes)으로 재밍·대역폭 감소가 있는 훈련을 모의하고 지연된 중요 전문을 대기열 적체·재전송 대기·링크 열화로 분류한 AAR 일지 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/c202_aar_report.md 및 configs/c202_tuned_params.yaml` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 오류 원인 규명의 객관성 및 개선 조치의 논리적 타당성

#### 미션 PASS 조건

- 정량 기준: AAR 피드백 적용 후 종합 정확도 10%p 이상 향상, 재발 오류 0건


- 진행 기준: AAR 보고서 작성 및 파라미터 튜닝 전후 회귀 테스트 통과


- 검증 방법: 회귀 테스트(Regression Test) 스위트 전체 자동 실행 및 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""AAR(사후검토) 일지 형식으로 중요 전문(FLASH·IMMEDIATE·PRIORITY)의 전송 지연 사례를 원인별로 분류하고, 큐 스케줄링 가중치(선점·aging)·조각 크기·재전송 타이머를 격자 탐색으로 재조정한 뒤 다른 운용 조건에서 회귀 테스트로 개선 전후를 비교한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

# ---- 우선순위 큐·재전송 이산 사건 시뮬레이터 공통 함수 ----
HDR = 32   # 전문 조각(패킷) 헤더 bytes (미션 2의 헤더 압축 적용 시 12 bytes)


def policy_key(m, t, policy, aging):
    # [TODO : 학생 실습 9] AAR 기반 중요 전문 전송 지연 사례 분석 및 큐 우선순위 가중치 재조정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def simulate(msgs, link, policy="strict", mtu=256, preempt=True, rto_mode="fixed", rto_s=4.0, aging=0.0,
             redundancy=False, adapt_mtu=False, hdr=HDR, seed=0, until=None, admit=None):
    """전문 목록을 링크로 보내는 이산 사건 시뮬레이션.
    - 전문은 mtu 크기 조각으로 나눠 보낸다. 조각이 손실되면 RTO 후 재전송한다(자료는 버리지 않음).
    - preempt=False면 한 전문의 조각을 끝까지 연속 전송한다(큰 전문이 긴급 전문을 막는 현상 재현).
    - rto_mode='adaptive'면 왕복시간(RTT) 추정값으로 RTO를 계산한다(Jacobson 방식).
    - redundancy=True면 링크 열화 추정 시 FLASH·IMMEDIATE 조각을 2회 연속 전송한다.
    - adapt_mtu=True면 링크 열화 추정 시 조각 크기를 줄인다.
    - admit(m, t)이 False인 전문은 새 조각을 보내지 않고 저장해 둔다(성능 저하 모드)."""
    rng = np.random.default_rng(seed)
    pend = sorted(({**m, "unsent": m["size"], "ready": [], "frags": [], "done": None, "first": None} for m in msgs), key=lambda m: m["t0"])
    t, i, active, cur = 0.0, 0, [], None
    srtt, rttvar, loss = None, None, 0.0
    st = {"tx": 0, "retx": 0, "spurious": 0, "bytes": 0, "decide_ms": [], "flag": {}, "log": []}
    until = until if until is not None else (pend[-1]["t0"] if pend else 0) + 6000
    while (i < len(pend) or active) and t < until:
        while i < len(pend) and pend[i]["t0"] <= t:
            active.append(pend[i]); i += 1
        td = time.perf_counter()
        for m in active:                                   # ACK가 이미 도착한 재전송 예약은 취소
            m["ready"] = [e for e in m["ready"] if not (e[1]["ack"] is not None and e[1]["ack"] <= t)]
        ready = [m for m in active if any(r <= t for r, _ in m["ready"]) or (m["unsent"] > 0 and (admit is None or admit(m, t)))]
        if not ready:
            nxt = ([pend[i]["t0"]] if i < len(pend) else []) + [r for m in active for r, _ in m["ready"] if r > t]
            t = min(nxt) if nxt else t + 1.0
            continue
        if not preempt and cur in ready and cur["unsent"] > 0:
            m = cur
        else:
            m = min(ready, key=lambda x: policy_key(x, t, policy, aging))
        st["decide_ms"].append((time.perf_counter() - td) * 1000)
        due = [e for e in m["ready"] if e[0] <= t]
        if due:
            e = min(due, key=lambda x: x[0]); m["ready"].remove(e); frag = e[1]
            st["retx"] += 1; st["spurious"] += frag["recv"]
        else:
            size = mtu if not adapt_mtu else (mtu if loss < 0.15 else (mtu // 2 if loss < 0.4 else mtu // 4))
            size = min(m["unsent"], size); m["unsent"] -= size
            frag = {"b": size, "recv": False, "ack": None, "tries": 0}
            m["frags"].append(frag)
        cur = m
        m["first"] = t if m["first"] is None else m["first"]
        nb = frag["b"] + hdr
        copies = 2 if (redundancy and RANK[m["prec"]] <= 1 and loss > 0.15) else 1
        for _ in range(copies):
            t += nb * 8 / link.bw(t)
            st["tx"] += 1; st["bytes"] += nb
            ok = rng.random() >= link.p_loss(nb, t)
            st["log"].append((t, nb, ok, m["prec"], m["id"]))            # 전송 기록(시각, bytes, 성공, 긴급도, 전문)
            loss = 0.85 * loss + 0.15 * (0.0 if ok else 1.0)          # 최근 손실률 추정(EWMA)
            st["flag"][int(t)] = st["flag"].get(int(t), False) or loss > 0.12   # 링크 열화 판정(초 단위)
            if ok and not frag["recv"]:
                frag["recv"], frag["rt"] = True, t
                r = link.rtt(rng); frag["ack"] = t + r
                if srtt is None:
                    srtt, rttvar = r, r / 2
                else:
                    rttvar = 0.75 * rttvar + 0.25 * abs(srtt - r); srtt = 0.875 * srtt + 0.125 * r
        frag["tries"] += 1
        rto = rto_s if (rto_mode == "fixed" or srtt is None) else float(np.clip(srtt + 4 * rttvar, 1.0, 30.0))
        if not frag["recv"] or frag["ack"] > t + rto:                   # 손실 또는 ACK 지연 -> 재전송 예약
            m["ready"].append((t + rto, frag))
        if m["done"] is None and m["unsent"] == 0 and all(f["recv"] for f in m["frags"]):
            m["done"] = max(f["rt"] for f in m["frags"])
        active = [x for x in active if not (x["unsent"] == 0 and not x["ready"] and all(f["recv"] for f in x["frags"]))]
    res = [{"id": m["id"], "type": m["type"], "prec": m["prec"], "size": m["size"], "t0": m["t0"], "done": m["done"], "wait": None if m["first"] is None else m["first"] - m["t0"],
            "retx": sum(f["tries"] for f in m["frags"]) - len(m["frags"]),
            "latency": None if m["done"] is None else m["done"] - m["t0"]} for m in pend]
    for r in res:
        r["on_time"] = r["latency"] is not None and r["latency"] <= DEADLINE[r["prec"]]
    return res, st


def summarize(res):
    """긴급도별 기한 내 전달률·지연. 종합 정확도 = 긴급도별 기한 내 전달률의 평균."""
    # [TODO : 학생 실습 9] AAR 기반 중요 전문 전송 지연 사례 분석 및 큐 우선순위 가중치 재조정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
INITIAL = dict(policy="strict", preempt=False, rto_mode="fixed", rto_s=6.0, mtu=512, aging=0.0)   # 초기 운용 설정


def aar_scenario(seed):
    """[교육용가상자료] 훈련 1회분: 재밍 60초 + 대역폭 50% 감소 200초 + 중계 지연이 큰 망."""
    msgs = make_messages(1200, 0.21, seed=seed)
    link = Link(12000, seed=seed + 100, jam=[(300, 360)], bw_drop=[(600, 800, 0.5)], rtt=(2.0, 0.8))
    return msgs, link


def aar_log(res, link):
    rows = []
    for r in res:
        if RANK[r["prec"]] <= 2 and not r["on_time"]:
            bad = any(link.state(k) != "good" or link.bw(k) < link.bw_bps for k in range(int(r["t0"]), int(r["t0"] + (r["latency"] or 60)) + 1))
            if r["retx"] > 0 and bad:
                cause = "링크 열화 중 재전송"
            elif r["retx"] > 0:
                cause = "재전송 타이머(RTO) 대기"
            elif r["wait"] is not None and r["wait"] > DEADLINE[r["prec"]] / 2:
                cause = "대기열 적체·하위 전문 점유"
            else:
                cause = "대용량 전문 분할 전송 지연"
            rows.append({"전문": r["id"], "긴급도": r["prec"], "크기": r["size"], "지연(s)": round(r["latency"] or -1, 1),
                         "첫 송신 대기(s)": round(r["wait"] or -1, 1), "재전송": r["retx"], "원인": cause})
    return pd.DataFrame(rows, columns=["전문", "긴급도", "크기", "지연(s)", "첫 송신 대기(s)", "재전송", "원인"])


# 1) 무엇이 일어났나: 초기 설정으로 훈련 운용
msgs, link = aar_scenario(7)
before_res, _ = simulate(msgs, link, **INITIAL)
tab0, acc0 = summarize(before_res)
log0 = aar_log(before_res, link)
print("[1. 무엇이 일어났나] 초기 설정 종합 정확도(긴급도별 기한 내 전달률 평균):", acc0)
print(tab0[["긴급도", "건수", "기한내 전달률", "95% 지연(s)"]].to_string(index=False))
print("\n[2. 왜 일어났나] 중요 전문 지연 원인 분류")
print(log0.groupby(["긴급도", "원인"]).size().rename("건수").to_string())

# 2) 무엇을 바꾸나: 격자 탐색(제약: FLASH 기한 내 전달률 95% 이상)
grid = []
for preempt in (False, True):
    for rto in ("fixed", "adaptive"):
        for mtu in (128, 256, 512):
            for aging in (0.0, 0.5, 1.0):
                cfg = dict(policy="aging" if aging else "strict", preempt=preempt, rto_mode=rto, rto_s=6.0, mtu=mtu, aging=aging)
                tab, acc = summarize(simulate(msgs, link, **cfg)[0])
                grid.append({**cfg, "FLASH": tab.loc[0, "기한내 전달률"], "종합정확도": acc})
g = pd.DataFrame(grid)
ok = g[g["FLASH"] >= 0.95].sort_values(["종합정확도", "FLASH"], ascending=False)
best = ok.iloc[0]
TUNED = dict(policy=best["policy"], preempt=bool(best["preempt"]), rto_mode=best["rto_mode"], rto_s=6.0, mtu=int(best["mtu"]), aging=float(best["aging"]))
print(f"\n[3. 무엇을 바꾸나] 격자 {len(g)}개 중 제약 충족 {len(ok)}개, 상위 3개")
print(ok.head(3)[["preempt", "rto_mode", "mtu", "aging", "FLASH", "종합정확도"]].to_string(index=False))

after_res, _ = simulate(msgs, link, **TUNED)
tab1, acc1 = summarize(after_res)
log1 = aar_log(after_res, link)
print(f"\n종합 정확도 {acc0:.3f} -> {acc1:.3f} ({(acc1 - acc0) * 100:+.1f}%p, 목표 +10%p: {'달성' if acc1 - acc0 >= 0.10 else '미달'})")
print(f"FLASH 기한 내 전달률 {tab0.loc[0, '기한내 전달률']:.1%} -> {tab1.loc[0, '기한내 전달률']:.1%}, 중요 전문 지연 {len(log0)}건 -> {len(log1)}건")
repeat = sorted(set(log0["전문"]) & set(log1["전문"]))
print(f"재보정 후에도 반복된 지연 전문: {repeat if repeat else '없음(재발 0건)'}")
for _, r in log1[log1["전문"].isin(repeat)].iterrows():
    print(f"  {r['전문']} {r['긴급도']} 지연 {r['지연(s)']}s, 재전송 {r['재전송']}회 -> 원인: {r['원인']} (스케줄링으로 줄이기 어려운 유형, 링크 열화 시 중복 전송 검토)")

# 3) 회귀 테스트: 다른 훈련 3회(다른 전문·링크)에서도 개선이 유지되는가
print("\n[회귀 테스트] 훈련  초기  재보정  FLASH(재보정)")
reg = []
for sd in (11, 12, 13):
    m2, l2 = aar_scenario(sd)
    a0 = summarize(simulate(m2, l2, **INITIAL)[0])[1]
    t1, a1 = summarize(simulate(m2, l2, **TUNED)[0])
    reg.append(a1 >= a0)
    print(f"             #{sd}   {a0:.3f}  {a1:.3f}   {t1.loc[0, '기한내 전달률']:.3f}")
print(f"회귀 테스트 {sum(reg)}/{len(reg)} 통과(재보정 후 종합 정확도가 떨어지지 않음)")

yaml = (f"scheduler:\n  policy: {TUNED['policy']}\n  preempt_fragment: {str(TUNED['preempt']).lower()}\n  aging: {TUNED['aging']}\n"
        f"fragment_mtu_bytes: {TUNED['mtu']}\nretransmission:\n  rto_mode: {TUNED['rto_mode']}\n  fixed_rto_s: 6.0\nconstraint:\n  min_flash_on_time: 0.95")
print("\n[재보정 파라미터(c202_tuned_params.yaml 내용)]\n" + yaml)
print("지속 개선: 훈련마다 AAR 일지를 누적해 같은 격자 탐색을 다시 수행하고, 변경 전 회귀 테스트 통과를 배포 조건으로 둔다.")

plt.figure(figsize=(7, 4))
for pre, c in ((False, "tab:gray"), (True, "tab:blue")):
    s = g[g["preempt"] == pre]
    plt.scatter(s["FLASH"], s["종합정확도"], c=c, label=f"fragment preempt={pre}")
plt.scatter([best["FLASH"]], [best["종합정확도"]], s=200, facecolors="none", edgecolors="red", label="selected")
plt.axvline(0.95, ls="--", c="gray"); plt.xlabel("FLASH on-time ratio"); plt.ylabel("overall accuracy (macro on-time)")
plt.legend(); plt.title("AAR re-tuning grid (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[1. 무엇이 일어났나] 초기 설정 종합 정확도(긴급도별 기한 내 전달률 평균): 0.921
      긴급도  건수  기한내 전달률  95% 지연(s)
    FLASH  25    0.720       19.4
IMMEDIATE  27    0.963       25.6
 PRIORITY  99    1.000       52.0
  ROUTINE 107    1.000      232.4

[2. 왜 일어났나] 중요 전문 지연 원인 분류
긴급도        원인             
FLASH      대기열 적체·하위 전문 점유    4
           링크 열화 중 재전송        3
IMMEDIATE  링크 열화 중 재전송        1

[3. 무엇을 바꾸나] 격자 36개 중 제약 충족 18개, 상위 3개
 preempt rto_mode  mtu  aging  FLASH  종합정확도
    True    fixed  128    0.0   0.96   0.99
    True    fixed  128    0.5   0.96   0.99
    True    fixed  128    1.0   0.96   0.99

종합 정확도 0.921 -> 0.990 (+6.9%p, 목표 +10%p: 미달)
FLASH 기한 내 전달률 72.0% -> 96.0%, 중요 전문 지연 8건 -> 1건
재보정 후에도 반복된 지연 전문: ['M0070']
  M0070 FLASH 지연 41.1s, 재전송 10회 -> 원인: 링크 열화 중 재전송 (스케줄링으로 줄이기 어려운 유형, 링크 열화 시 중복 전송 검토)

[회귀 테스트] 훈련  초기  재보정  FLASH(재보정)
             #11   0.936  1.000   1.000
             #12   0.953  0.988   0.952
             #13   0.943  0.951   0.870
회귀 테스트 3/3 통과(재보정 후 종합 정확도가 떨어지지 않음)

[재보정 파라미터(c202_tuned_params.yaml 내용)]
scheduler:
  policy: strict
  preempt_fragment: true
  aging: 0.0
fragment_mtu_bytes: 128
retransmission:
  rto_mode: fixed
  fixed_rto_s: 6.0
constraint:
... (이하 생략)
```
</details>
